# Flex travel allocation - input data audit
Describes and audits every raw input table before allocation. For each table it
reports a **data sample**, **column data types**, **null / distinct counts** and a
**statistical description**. Tables are read as delivered (no cleaning) so the audit
reflects the real inputs. Read-only: it never writes and never mutates the data.

In [9]:
# Setup
import os
import sys

import pyspark.sql.functions as sf
from pyspark.sql import DataFrame, SparkSession

from conf.config import ENVIRONMENT_NON_PRODUCTION, Table
from flex_travel_allocation.data_handler import DataHandler

# Databricks provides Spark; locally (e.g. Jupyter on Windows) the Python worker needs
# PYSPARK_PYTHON and a stable loopback address to connect back to the driver.
if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    spark = SparkSession.builder.getOrCreate()
else:
    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
    os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
    spark = (
        SparkSession.builder.appName("flex-input-audit")
        .master("local[*]")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )

try:
    environment = dbutils.widgets.get("environment")  # type: ignore[name-defined]
except Exception:
    environment = ENVIRONMENT_NON_PRODUCTION

data_handler = DataHandler.from_config(spark=spark, environment=environment)

SAMPLE_ROWS = 20

INPUT_TABLES = [
    Table.ARTICLES,
    Table.SALES,
    Table.ACTIVATIONS_ARTICLE,
    Table.ACTIVATIONS,
    Table.MINUTES_TO_ADD_PROBABILITIES,
    Table.HOURS_BETWEEN_PROBABILITIES,
]

In [10]:
# Audit helpers - use display() on Databricks, fall back to .show() locally


def render(dataframe: DataFrame, n: int = SAMPLE_ROWS) -> None:
    # Collect the (already small) result to pandas so real values render as a table
    # in Jupyter/Databricks; plain Spark display() only shows the schema repr.
    pdf = dataframe.limit(n).toPandas()
    try:
        display(pdf)  # type: ignore[name-defined]
    except NameError:
        print(pdf.to_string(index=False))


def schema_overview(dataframe: DataFrame) -> DataFrame:
    """One row per column: name, Spark data type and nullable flag."""
    rows = [(f.name, f.dataType.simpleString(), f.nullable) for f in dataframe.schema.fields]
    return spark.createDataFrame(rows, ["column", "data_type", "nullable"])


def null_and_distinct_counts(dataframe: DataFrame) -> DataFrame:
    """Per-column non-null, null and distinct counts."""
    total = dataframe.count()
    exprs = []
    for column in dataframe.columns:
        exprs.append(sf.count(sf.col(column)).alias(f"{column}__nonnull"))
        exprs.append(sf.sum(sf.col(column).isNull().cast("long")).alias(f"{column}__null"))
        exprs.append(sf.countDistinct(sf.col(column)).alias(f"{column}__distinct"))
    agg = dataframe.agg(*exprs).collect()[0].asDict()
    rows = [
        (column, total, agg[f"{column}__nonnull"], agg[f"{column}__null"], agg[f"{column}__distinct"])
        for column in dataframe.columns
    ]
    return spark.createDataFrame(rows, ["column", "rows", "non_null", "nulls", "distinct"])


def descriptive_statistics(dataframe: DataFrame) -> DataFrame:
    """One row per numeric column: count, nulls, mean, stddev, min, quartiles, median, max, skewness, kurtosis."""
    numeric_types = {"tinyint", "smallint", "int", "bigint", "float", "double", "decimal"}
    numeric_columns = [
        field.name
        for field in dataframe.schema.fields
        if field.dataType.simpleString().split("(")[0] in numeric_types
    ]
    metrics = ["count", "nulls", "mean", "stddev", "min", "p25", "median", "p75", "max", "skewness", "kurtosis"]
    schema = (
        "column string, count long, nulls long, mean double, stddev double, "
        "min double, p25 double, median double, p75 double, max double, "
        "skewness double, kurtosis double"
    )
    if not numeric_columns:
        return spark.createDataFrame([], schema)
    exprs = []
    for column in numeric_columns:
        quoted = f"`{column}`"
        exprs += [
            sf.count(sf.col(column)).alias(f"{column}~count"),
            sf.sum(sf.col(column).isNull().cast("long")).alias(f"{column}~nulls"),
            sf.mean(sf.col(column)).alias(f"{column}~mean"),
            sf.stddev(sf.col(column)).alias(f"{column}~stddev"),
            sf.min(sf.col(column)).cast("double").alias(f"{column}~min"),
            sf.expr(f"percentile_approx({quoted}, 0.25)").cast("double").alias(f"{column}~p25"),
            sf.expr(f"percentile_approx({quoted}, 0.5)").cast("double").alias(f"{column}~median"),
            sf.expr(f"percentile_approx({quoted}, 0.75)").cast("double").alias(f"{column}~p75"),
            sf.max(sf.col(column)).cast("double").alias(f"{column}~max"),
            sf.skewness(sf.col(column)).alias(f"{column}~skewness"),
            sf.kurtosis(sf.col(column)).alias(f"{column}~kurtosis"),
        ]
    values = dataframe.agg(*exprs).collect()[0].asDict()
    rows = []
    for column in numeric_columns:
        rows.append(
            (
                column,
                int(values[f"{column}~count"]) if values[f"{column}~count"] is not None else None,
                int(values[f"{column}~nulls"]) if values[f"{column}~nulls"] is not None else None,
                *[
                    float(values[f"{column}~{metric}"]) if values[f"{column}~{metric}"] is not None else None
                    for metric in metrics[2:]
                ],
            )
        )
    return spark.createDataFrame(rows, schema)


def non_numeric_statistics(dataframe: DataFrame) -> DataFrame:
    """One row per non-numeric column (text/date/boolean): count, nulls, distinct, min, max, mode."""
    numeric_types = {"tinyint", "smallint", "int", "bigint", "float", "double", "decimal"}
    categorical = [
        field.name
        for field in dataframe.schema.fields
        if field.dataType.simpleString().split("(")[0] not in numeric_types
    ]
    schema = (
        "column string, data_type string, count long, nulls long, distinct long, "
        "min string, max string, top string, top_freq long"
    )
    if not categorical:
        return spark.createDataFrame([], schema)
    types = {field.name: field.dataType.simpleString() for field in dataframe.schema.fields}
    exprs = []
    for column in categorical:
        exprs += [
            sf.count(sf.col(column)).alias(f"{column}~count"),
            sf.sum(sf.col(column).isNull().cast("long")).alias(f"{column}~nulls"),
            sf.countDistinct(sf.col(column)).alias(f"{column}~distinct"),
            sf.min(sf.col(column)).cast("string").alias(f"{column}~min"),
            sf.max(sf.col(column)).cast("string").alias(f"{column}~max"),
        ]
    values = dataframe.agg(*exprs).collect()[0].asDict()
    rows = []
    for column in categorical:
        mode = (
            dataframe.filter(sf.col(column).isNotNull())
            .groupBy(sf.col(column).cast("string").alias("value"))
            .count()
            .orderBy(sf.desc("count"), "value")
            .first()
        )
        top, top_freq = (mode["value"], int(mode["count"])) if mode is not None else (None, None)
        rows.append(
            (
                column,
                types[column],
                int(values[f"{column}~count"]) if values[f"{column}~count"] is not None else None,
                int(values[f"{column}~nulls"]) if values[f"{column}~nulls"] is not None else None,
                int(values[f"{column}~distinct"]) if values[f"{column}~distinct"] is not None else None,
                values[f"{column}~min"],
                values[f"{column}~max"],
                top,
                top_freq,
            )
        )
    return spark.createDataFrame(rows, schema)


def audit_table(table_name: str) -> None:
    dataframe = data_handler.read_table(table_name)
    print(f"TABLE: {table_name}  |  rows: {dataframe.count():,}  |  columns: {len(dataframe.columns)}")
    print("--- column data types ---")
    render(schema_overview(dataframe), n=len(dataframe.columns))
    print("--- data sample ---")
    render(dataframe, n=SAMPLE_ROWS)
    print("--- null / distinct counts ---")
    render(null_and_distinct_counts(dataframe), n=len(dataframe.columns))
    print("--- descriptive statistics (numeric) ---")
    render(descriptive_statistics(dataframe), n=len(dataframe.columns))
    print("--- descriptive statistics (non-numeric) ---")
    render(non_numeric_statistics(dataframe), n=len(dataframe.columns))

## 1. Articles (`article_code_flex`)
Product reference: article code -> Flex product, max activations and validity window.

In [11]:
audit_table(Table.ARTICLES)

TABLE: article_code_flex  |  rows: 4  |  columns: 3
--- column data types ---


,column,data_type,nullable
0,article_code_int,smallint,True
1,article_description_nl,string,True
2,maximum_activations_allowed,int,True


--- data sample ---


,article_code_int,article_description_nl,maximum_activations_allowed
0,256,Flex Abonnement 6,6
1,258,Flex Abonnement 80,80
2,257,Flex Abonnement 10,10
3,259,Flex Abonnement 120,120


--- null / distinct counts ---


,column,rows,non_null,nulls,distinct
0,article_code_int,4,4,0,4
1,article_description_nl,4,4,0,4
2,maximum_activations_allowed,4,4,0,4


--- descriptive statistics (numeric) ---


,column,count,nulls,mean,stddev,min,p25,median,p75,max,skewness,kurtosis
0,article_code_int,4,0,257.5,1.290994,256.0,256.0,257.0,258.0,259.0,0.000000,-1.360000
1,maximum_activations_allowed,4,0,54.0,55.593765,6.0,6.0,10.0,80.0,120.0,0.244835,-1.674504


--- descriptive statistics (non-numeric) ---


,column,data_type,count,nulls,distinct,min,max,top,top_freq
0,article_description_nl,string,4,0,4,Flex Abonnement 10,Flex Abonnement 80,Flex Abonnement 10,1


## 2. Sales (`sales_flex_dummy`)
One row per Flex subscription sold: ticket number, article, net sales and travel dates.

In [12]:
audit_table(Table.SALES)

TABLE: sales_flex  |  rows: 37,675  |  columns: 6
--- column data types ---


,column,data_type,nullable
0,net_sales_euro,double,True
1,sales_date,date,True
2,ticket_number,bigint,True
3,travel_start_date,date,True
4,travel_end_date,date,True
5,article_description_nl,string,True


--- data sample ---


,net_sales_euro,sales_date,ticket_number,travel_start_date,travel_end_date,article_description_nl
0,95.96,2025-01-01,3533611089227163,2025-01-01,2025-01-31,Flex Abonnement 6
1,81.98,2025-01-01,2010491083439785399,2025-01-01,2025-01-31,Flex Abonnement 6
2,90.88,2025-01-01,7967589733060493850,2025-01-01,2025-01-31,Flex Abonnement 6
3,96.59,2025-01-01,106026541225706019,2025-01-01,2025-01-31,Flex Abonnement 6
4,84.43,2025-01-01,3787637304078836262,2025-01-01,2025-01-31,Flex Abonnement 6
5,85.77,2025-01-01,8323148856480699395,2025-01-01,2025-01-31,Flex Abonnement 6
6,91.78,2025-01-01,3772969875237455748,2025-01-01,2025-01-31,Flex Abonnement 6
7,88.84,2025-01-01,5811459011332620322,2025-01-01,2025-01-31,Flex Abonnement 6
8,96.71,2025-01-01,7350390037566659275,2025-01-01,2025-01-31,Flex Abonnement 6
9,82.11,2025-01-01,7516720083082398004,2025-01-01,2025-01-31,Flex Abonnement 6


--- null / distinct counts ---


,column,rows,non_null,nulls,distinct
0,net_sales_euro,37675,37675,0,16815
1,sales_date,37675,37675,0,64
2,ticket_number,37675,37675,0,37675
3,travel_start_date,37675,37675,0,97
4,travel_end_date,37675,37675,0,194
5,article_description_nl,37675,37675,0,4


--- descriptive statistics (numeric) ---


,column,count,nulls,mean,stddev,min,p25,median,p75,max,skewness,kurtosis
0,net_sales_euro,37675,0,5.543349e+02,5.632914e+02,7.746000e+01,9.259000e+01,1.224400e+02,1.146990e+03,1.751770e+03,0.513509,-1.585486
1,ticket_number,37675,0,4.628494e+18,2.670272e+18,1.176459e+14,2.306612e+18,4.641514e+18,6.956451e+18,9.223036e+18,-0.003777,-1.204840


--- descriptive statistics (non-numeric) ---


,column,data_type,count,nulls,distinct,min,max,top,top_freq
0,sales_date,date,37675,0,64,2025-01-01,2025-03-31,2025-02-07,1108
1,travel_start_date,date,37675,0,97,2025-01-01,2025-04-07,2025-02-07,1071
2,travel_end_date,date,37675,0,194,2025-01-31,2026-04-06,2025-03-09,756
3,article_description_nl,string,37675,0,4,Flex Abonnement 10,Flex Abonnement 80,Flex Abonnement 10,11574


## 3. Activations per article (`activations_article_dummy`)
Links each activation ticket number to its article code.

In [13]:
audit_table(Table.ACTIVATIONS_ARTICLE)

TABLE: activations_article  |  rows: 37,675  |  columns: 2
--- column data types ---


,column,data_type,nullable
0,article_code_int,smallint,True
1,ticket_number_activations,string,True


--- data sample ---


,article_code_int,ticket_number_activations
0,256,20251283653779531431918
1,256,20250726054928625828292
2,256,20258669372709628445507
3,256,20255127713658893062524
4,256,20255012964552157262831
5,256,20250189464267681508016
6,256,20252295076489150532297
7,256,20258809379763763631450
8,256,20257650310339219607559
9,256,20258319874317310779977


--- null / distinct counts ---


,column,rows,non_null,nulls,distinct
0,article_code_int,37675,37675,0,4
1,ticket_number_activations,37675,37675,0,37675


--- descriptive statistics (numeric) ---


,column,count,nulls,mean,stddev,min,p25,median,p75,max,skewness,kurtosis
0,article_code_int,37675,0,257.232701,1.009917,256.0,256.0,257.0,258.0,259.0,0.26274,-1.064899


--- descriptive statistics (non-numeric) ---


,column,data_type,count,nulls,distinct,min,max,top,top_freq
0,ticket_number_activations,string,37675,0,37675,20250000117645872809217,20259223036066819289782,20250000117645872809217,1


## 4. Activations (`activations_flex_dummy`)
Individual activation (usage) events - the largest input table.

In [14]:
audit_table(Table.ACTIVATIONS)

TABLE: activations_flex  |  rows: 1,093,237  |  columns: 6
--- column data types ---


,column,data_type,nullable
0,ticket_number_activations,string,True
1,activation_date,date,True
2,activation_sequence,int,True
3,travel_start_timestamp_cet,timestamp,True
4,travel_end_timestamp_cet,timestamp,True
5,activation_timestamp_utc,timestamp,True


--- data sample ---


,ticket_number_activations,activation_date,activation_sequence,travel_start_timestamp_cet,travel_end_timestamp_cet,activation_timestamp_utc
0,20250000117645872809217,2025-02-04,1,2025-02-04 01:00:00,2025-02-05 00:59:59,2025-02-04 07:48:47
1,20250000117645872809217,2025-02-06,2,2025-02-07 01:00:00,2025-02-08 00:59:59,2025-02-06 15:39:52
2,20250000117645872809217,2025-02-11,3,2025-02-11 01:00:00,2025-02-12 00:59:59,2025-02-11 06:15:30
3,20250000117645872809217,2025-02-14,4,2025-02-14 01:00:00,2025-02-15 00:59:59,2025-02-14 07:12:47
4,20250000117645872809217,2025-02-17,5,2025-02-17 01:00:00,2025-02-18 00:59:59,2025-02-17 06:38:44
5,20250000117645872809217,2025-02-19,6,2025-02-19 01:00:00,2025-02-20 00:59:59,2025-02-19 08:51:16
6,20250000117645872809217,2025-02-26,7,2025-02-26 01:00:00,2025-02-27 00:59:59,2025-02-26 07:49:16
7,20250000117645872809217,2025-02-27,8,2025-02-28 01:00:00,2025-03-01 00:59:59,2025-02-27 08:45:50
8,20250000117645872809217,2025-03-04,9,2025-03-04 01:00:00,2025-03-05 00:59:59,2025-03-04 08:20:14
9,20250000117645872809217,2025-03-05,10,2025-03-05 01:00:00,2025-03-06 00:59:59,2025-03-05 09:04:57


--- null / distinct counts ---


,column,rows,non_null,nulls,distinct
0,ticket_number_activations,1093237,1093237,0,37675
1,activation_date,1093237,1093237,0,328
2,activation_sequence,1093237,1093237,0,109
3,travel_start_timestamp_cet,1093237,1093237,0,328
4,travel_end_timestamp_cet,1093237,1093237,0,328
5,activation_timestamp_utc,1093237,1093237,0,968545


--- descriptive statistics (numeric) ---


,column,count,nulls,mean,stddev,min,p25,median,p75,max,skewness,kurtosis
0,activation_sequence,1093237,0,28.727424,22.132355,1.0,8.0,25.0,45.0,109.0,0.645213,-0.321989


--- descriptive statistics (non-numeric) ---


,column,data_type,count,nulls,distinct,min,max,top,top_freq
0,ticket_number_activations,string,1093237,0,37675,20250000117645872809217,20259223036066819289782,20250121033222429496159,109
1,activation_date,date,1093237,0,328,2025-01-01,2026-04-03,2025-03-26,6366
2,travel_start_timestamp_cet,timestamp,1093237,0,328,2025-01-01 01:00:00,2026-04-03 02:00:00,2025-03-28 01:00:00,7002
3,travel_end_timestamp_cet,timestamp,1093237,0,328,2025-01-02 00:59:59,2026-04-04 01:59:59,2025-03-29 00:59:59,7002
4,activation_timestamp_utc,timestamp,1093237,0,968545,2025-01-01 05:05:49,2026-04-03 08:49:46,2025-02-27 07:13:54,6


## 5. Minutes-to-add probabilities (`minutes_to_add_probabilities`)
Weighted distribution of the minute offset added when sampling travel times.

In [15]:
audit_table(Table.MINUTES_TO_ADD_PROBABILITIES)

TABLE: minutes_to_add_probabilities  |  rows: 8  |  columns: 2
--- column data types ---


,column,data_type,nullable
0,minutes_to_add,int,True
1,probability,double,True


--- data sample ---


,minutes_to_add,probability
0,-10,0.18
1,0,0.19
2,5,0.22
3,15,0.14
4,30,0.16
5,60,0.08
6,120,0.02
7,180,0.01


--- null / distinct counts ---


,column,rows,non_null,nulls,distinct
0,minutes_to_add,8,8,0,8
1,probability,8,8,0,8


--- descriptive statistics (numeric) ---


,column,count,nulls,mean,stddev,min,p25,median,p75,max,skewness,kurtosis
0,minutes_to_add,8,0,50.000,67.241569,-10.00,0.00,15.00,60.00,180.00,1.033784,-0.327793
1,probability,8,0,0.125,0.079282,0.01,0.02,0.14,0.18,0.22,-0.436698,-1.303492


--- descriptive statistics (non-numeric) ---


,column,data_type,count,nulls,distinct,min,max,top,top_freq


## 6. Hours-between probabilities (`hours_between_there_and_back_probabilities`)
Weighted distribution of the hours between the there and back travels.

In [16]:
audit_table(Table.HOURS_BETWEEN_PROBABILITIES)

TABLE: hours_between_there_and_back_probabilities  |  rows: 24  |  columns: 2
--- column data types ---


,column,data_type,nullable
0,hours_between_there_and_back,double,True
1,probability,double,True


--- data sample ---


,hours_between_there_and_back,probability
0,0.0,0.00
1,1.0,0.00
2,2.0,0.01
3,3.0,0.04
4,4.0,0.04
5,5.0,0.07
6,6.0,0.09
7,7.0,0.08
8,8.0,0.11
9,9.0,0.22


--- null / distinct counts ---


,column,rows,non_null,nulls,distinct
0,hours_between_there_and_back,24,24,0,24
1,probability,24,24,0,10


--- descriptive statistics (numeric) ---


,column,count,nulls,mean,stddev,min,p25,median,p75,max,skewness,kurtosis
0,hours_between_there_and_back,24,0,11.500000,7.071068,0.0,5.0,11.00,17.00,23.00,0.000000,-1.204174
1,probability,24,0,0.041667,0.061479,0.0,0.0,0.01,0.07,0.22,1.580486,1.652654


--- descriptive statistics (non-numeric) ---


,column,data_type,count,nulls,distinct,min,max,top,top_freq
